# MuSeg — Evaluation vs Ground Truth (Water / Fat Fraction / Dixon)

Computes per-muscle metrics for MuSeg segmentations against the myosegmenTUM
ground truth across all three modalities.

| Section | Seg dir | File suffix | Result dir | CSV suffix |
|---|---|---|---|---|
| Water | `segs_water/` | `_WATER_stack{N}_museg.nii.gz` | `results_water/` | `museg_water` |
| Fat Fraction | `segmentation_fat_frac/` | `_FATFRACTION_stack{N}_museg.nii.gz` | `results_fat_frac/` | `museg_fatfrac` |
| Dixon | `segmentation_dixon/` | `_stack{N}_museg.nii.gz` | `results_dixon/` | `museg_dixon` |

**MuSeg label mapping** (no L/R distinction — both sides share one label):

| Muscle | MuSeg label | GT label |
|---|---|---|
| R_gracilis | 6 | 5 |
| L_gracilis | 6 | 1 |
| R_sartorius | 5 | 8 |
| L_sartorius | 5 | 4 |

In [1]:
import glob
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [2]:
BOUNDARY_DISTANCE = 1

EVAL_DIR = r'C:\Projects\dissector\eval_notebooks'
BASE_DIR = os.path.join(EVAL_DIR, 'museg')
GT_BASE  = os.path.join(EVAL_DIR, 'myosegmenTUM')

SEG_DIR_WATER  = os.path.join(BASE_DIR, 'segs_water')
SEG_DIR_FF     = os.path.join(BASE_DIR, 'segmentation_fat_frac')
SEG_DIR_DIXON  = os.path.join(BASE_DIR, 'segmentation_dixon')

# (muscle_name, gt_label_int, museg_label_int)
# MuSeg assigns one label per muscle type with no L/R distinction.
MUSCLES = [
    ('R_gracilis',  5, 6),
    ('L_gracilis',  1, 6),
    ('R_sartorius', 8, 5),
    ('L_sartorius', 4, 5),
]

for label, path in [
    ('GT_BASE',       GT_BASE),
    ('Water segs',    SEG_DIR_WATER),
    ('Fat frac segs', SEG_DIR_FF),
    ('Dixon segs',    SEG_DIR_DIXON),
]:
    print(f'{label}: {path}  exists={os.path.isdir(path)}')

GT_BASE: C:\Projects\dissector\eval_notebooks\myosegmenTUM  exists=True
Water segs: C:\Projects\dissector\eval_notebooks\museg\segs_water  exists=True
Fat frac segs: C:\Projects\dissector\eval_notebooks\museg\segmentation_fat_frac  exists=True
Dixon segs: C:\Projects\dissector\eval_notebooks\museg\segmentation_dixon  exists=True


In [3]:
def parse_subject_stack_modality(filename, modality_tag):
    """
    Extract (subject, stack_num) from filenames that carry a modality tag:
      HV001_1_WATER_stack1_museg.nii.gz
      P004_1_FATFRACTION_stack2_museg.nii.gz
    Returns (None, None) if the pattern does not match.
    """
    pattern = rf'^(.+)_{modality_tag}_stack(\d+)_museg\.nii\.gz$'
    m = re.match(pattern, os.path.basename(filename))
    if not m:
        return None, None
    return m.group(1), m.group(2)


def parse_subject_stack_dixon(filename):
    """
    Extract (subject, stack_num) from Dixon filenames (no modality tag):
      HV001_1_stack1_museg.nii.gz
      P004_1_stack2_museg.nii.gz
    Returns (None, None) if the pattern does not match.
    """
    pattern = r'^(.+)_stack(\d+)_museg\.nii\.gz$'
    m = re.match(pattern, os.path.basename(filename))
    if not m:
        return None, None
    return m.group(1), m.group(2)


def evaluate_muscle(muscle_name, gt_label_idx, museg_label_idx,
                    seg_files, parse_fn, result_dir, csv_suffix):
    results = []
    for seg_file in seg_files:
        subject, stack_num = parse_fn(seg_file)
        if subject is None:
            print(f'  could not parse: {os.path.basename(seg_file)}, skipping')
            continue

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        seg_sitk  = sitk.ReadImage(seg_file)
        pred_arr  = (sitk.GetArrayFromImage(seg_sitk) == museg_label_idx).astype(np.uint8)
        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {os.path.basename(seg_file)}: empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'subject':                              subject,
            'stack':                                stack_num,
            'pred_file':                            os.path.basename(seg_file),
            'gt_path':                              gt_path,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(results)
    os.makedirs(result_dir, exist_ok=True)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_{csv_suffix}.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df


print('Functions ready.')

Functions ready.


## Water

In [4]:
RESULT_DIR_WATER = os.path.join(BASE_DIR, 'results_water')

seg_files_water = sorted(glob.glob(os.path.join(SEG_DIR_WATER, '*_museg.nii.gz')))
print(f'Found {len(seg_files_water)} water files')
for f in seg_files_water[:4]:
    print(' ', os.path.basename(f))

Found 46 water files
  HV001_1_WATER_stack1_museg.nii.gz
  HV001_1_WATER_stack2_museg.nii.gz
  HV001_2_WATER_stack1_museg.nii.gz
  HV001_2_WATER_stack2_museg.nii.gz


In [ ]:
dfs_water = {}
for muscle_name, gt_idx, museg_idx in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, museg={museg_idx}) ──')
    dfs_water[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, museg_idx,
        seg_files_water,
        lambda f: parse_subject_stack_modality(f, 'WATER'),
        RESULT_DIR_WATER, 'museg_water',
    )
print('\nDone.')


── R_gracilis (gt=5, museg=6) ──


In [ ]:
for name, df in dfs_water.items():
    print(f'\n── {name} ──')
    display(df[['subject', 'stack', f'{name}_dice', f'{name}_hausdorff',
                f'{name}_jaccard', f'{name}_boundary_iou_3d']].head(10))

## Fat Fraction

In [ ]:
RESULT_DIR_FF = os.path.join(BASE_DIR, 'results_fat_frac')

seg_files_ff = sorted(glob.glob(os.path.join(SEG_DIR_FF, '*_museg.nii.gz')))
print(f'Found {len(seg_files_ff)} fat fraction files')
for f in seg_files_ff[:4]:
    print(' ', os.path.basename(f))

In [ ]:
dfs_ff = {}
for muscle_name, gt_idx, museg_idx in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, museg={museg_idx}) ──')
    dfs_ff[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, museg_idx,
        seg_files_ff,
        lambda f: parse_subject_stack_modality(f, 'FATFRACTION'),
        RESULT_DIR_FF, 'museg_fatfrac',
    )
print('\nDone.')

In [ ]:
for name, df in dfs_ff.items():
    print(f'\n── {name} ──')
    display(df[['subject', 'stack', f'{name}_dice', f'{name}_hausdorff',
                f'{name}_jaccard', f'{name}_boundary_iou_3d']].head(10))

## Dixon

Dixon files have no WATER/FATFRACTION tag: `{subject}_stack{N}_museg.nii.gz`

In [ ]:
RESULT_DIR_DIXON = os.path.join(BASE_DIR, 'results_dixon')

seg_files_dixon = sorted(glob.glob(os.path.join(SEG_DIR_DIXON, '*_museg.nii.gz')))
print(f'Found {len(seg_files_dixon)} Dixon files')
for f in seg_files_dixon[:4]:
    print(' ', os.path.basename(f))

In [ ]:
dfs_dixon = {}
for muscle_name, gt_idx, museg_idx in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, museg={museg_idx}) ──')
    dfs_dixon[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, museg_idx,
        seg_files_dixon,
        parse_subject_stack_dixon,
        RESULT_DIR_DIXON, 'museg_dixon',
    )
print('\nDone.')

In [ ]:
for name, df in dfs_dixon.items():
    print(f'\n── {name} ──')
    display(df[['subject', 'stack', f'{name}_dice', f'{name}_hausdorff',
                f'{name}_jaccard', f'{name}_boundary_iou_3d']].head(10))